# tiny-voca Sinhala — 1. Data (CPU)

Downloads Sinhala speech, converts it to 24 kHz and computes the acoustic features
that training needs. **Runs on CPU only, so it does not use your GPU hours.**

**Before you run**
1. Settings (right panel): **Accelerator: None**, **Internet: On** (needs a phone-verified Kaggle account).
2. Click **Save Version → Save & Run All (Commit)**. It runs in the background, about 1–2 hours.
   You can close the browser.
3. When it finishes, open notebook 2 and add this notebook's output as its input.

**Data licences:** Path Nirvana recordings are GPL-3.0 and the speakers allow only
non-obscene, non-offensive speech generation. OpenSLR-30 is CC BY-SA 4.0.

In [ ]:
# Settings
USE_PATHNIRVANA_V2 = True    # ~13.8 h: male M1 (Ven. Mettananda) + female F1 (Mrs. Oshadi)
USE_PATHNIRVANA_OLD = True   # ~7 h more of M1 (the first release)
USE_OPENSLR30 = False        # Google's Sinhala TTS corpus: extra voices S<id> (~3 h in total)
MAX_PER_VOICE = None         # e.g. 500 for a quick trial run

In [ ]:
# Get tiny-voca-sinhala (tiny-voca with Sinhala support) at a fixed version and install what is missing.
import os, subprocess, sys, json, time, shutil, glob
from pathlib import Path
os.environ["TINY_VOCA_LANGUAGE"] = "si"          # switches tiny-voca to the Sinhala inventory
REPO = Path("/tmp/tiny-voca-sinhala")
COMMIT = "a1fc282ed60eb0294b5d352ea8c49e50412e417c"

def sh(command, quiet=False):
    """Run a shell command, show its output, stop the notebook if it fails."""
    print("$", command, flush=True)
    result = subprocess.run(command, shell=True, text=True, capture_output=quiet)
    if result.returncode:
        if quiet:
            print(result.stdout[-3000:], result.stderr[-3000:])
        raise SystemExit(f"command failed ({result.returncode}): {command}")

if not REPO.exists():
    sh(f"git clone -q https://github.com/Emindu/tiny-voca-sinhala {REPO}")
    sh(f"git -C {REPO} checkout -q {COMMIT}")
sh('pip install -q "phonemizer-fork==3.3.2" "espeakng-loader==0.2.4"', quiet=True)
sh(f"pip install -q --no-deps -e {REPO}", quiet=True)
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONPATH"] = f"{REPO / 'src'}:{REPO / 'tests'}"
# "discover -s tests" imports the test as a top-level module: Kaggle has an unrelated package named "tests".
check = subprocess.run(f"cd {REPO} && python -m unittest discover -s tests -p test_frontend_sinhala.py",
                       shell=True, capture_output=True, text=True)
print(check.stderr[-1500:])
if check.returncode:
    raise SystemExit("the Sinhala front-end tests failed (see above)")

In [ ]:
# Download (to /tmp, so the raw files are not saved as output)
RAW = Path("/tmp/raw")
script = REPO / "scripts" / "sinhala_data.py"
sources = []
if USE_PATHNIRVANA_V2:
    try:
        sh(f"python {script} download --source pathnirvana --dest {RAW}")
        sources.append(str(RAW / "pathnirvana"))
    except SystemExit:
        print("Could not get the v2 release; using the first release only.")
        USE_PATHNIRVANA_OLD = True
if USE_PATHNIRVANA_OLD:
    sh(f"python {script} download --source pathnirvana-old --dest {RAW}")
    sources.append(str(RAW / "pathnirvana-old" / "old dataset"))
openslr = []
if USE_OPENSLR30:
    sh(f"python {script} download --source openslr30 --dest {RAW}")
    openslr = [str(RAW / "openslr30")]

In [ ]:
# Convert to 24 kHz FLAC Parquet shards, split by sentence
DATA = Path("/tmp/data")
command = f"python {script} build --output {DATA}"
if sources:
    command += " --pathnirvana " + " ".join(f'"{s}"' for s in sources)
if openslr:
    command += " --openslr30 " + " ".join(openslr)
if MAX_PER_VOICE:
    command += f" --max-per-voice {MAX_PER_VOICE}"
sh(command)
shutil.rmtree(RAW, ignore_errors=True)        # free disk space

In [ ]:
# Acoustic features (waveform, mel, pitch, energy): one split at a time, every CPU core working on it
OUT = Path("/kaggle/working")
FEATURES = OUT / "features"
shutil.copytree(DATA, OUT / "data", ignore=shutil.ignore_patterns("*.parquet"), dirs_exist_ok=True)  # config + summary
config = OUT / "data" / "config.json"
workers = os.cpu_count() or 4
single = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "NUMBA_NUM_THREADS": "1"}
for split in ("train", "validation", "test"):
    shards = sorted(glob.glob(str(DATA / f"{split}-*.parquet")))
    if not shards:
        raise SystemExit(f"the {split} split has no clips: with MAX_PER_VOICE set this low, raise it (or set None)")
    log = open(f"/tmp/prepare-{split}.log", "w")
    job = subprocess.Popen([sys.executable, "-m", "tiny_voca.prepare", "--inputs", *shards, "--split", split, "--output", str(FEATURES),
                            "--config", str(config), "--workers", str(workers)], stdout=log, stderr=subprocess.STDOUT, env=single)
    start = time.time()
    while job.poll() is None:
        time.sleep(60)
        done = sum(1 for _ in open(FEATURES / f"{split}.jsonl")) if (FEATURES / f"{split}.jsonl").exists() else 0
        print(f"{split}: {(time.time() - start) / 60:5.1f} min, {done} clips done", flush=True)
    log.close()
    print(split, open(log.name).read().strip().splitlines()[-1], flush=True)
    if job.returncode:
        raise SystemExit(f"prepare {split} failed; see /tmp/prepare-{split}.log")
for lock in FEATURES.glob("*.lock"):
    lock.unlink()
print(open(OUT / "data" / "summary.json").read())
sh(f"du -sh {FEATURES}")
print("Done. Now open notebook 2 and add this notebook's output as input.")